# SentryScribe: Colab runner

This notebook runs the **computer-vision half** of SentryScribe on a Colab GPU and then the **LLM half** on top of its output.

| Step | What it does | Needs |
|---|---|---|
| 1 | Install libraries, check the GPU | A100 GPU runtime (T4 works, with a smaller LLM) |
| 2 | Load your CCTV clips, or skip this in benchmark mode | Upload or Google Drive |
| 3 | Set zones, lines and thresholds | Edit one cell |
| 3b | **Benchmark mode** (on by default): download real, hand-annotated footage (9 sequences of MOT15 and MOT16) and build its zones and ground truth | Internet, about 3.2 GB |
| 4–5 | Load the pipeline code, preview zones on frame 1 | |
| 6 | **Process the videos**: YOLOv8 → ByteTrack → event rules (YOLOv8x at 1280 px on an A100) | about 1–3 min per clip |
| 7 | Inspect events, snapshots and the annotated video | |
| 8 | **Evaluate** against `ground_truth.csv` (made in 3b, or hand-labelled for your own clips) and write `metrics.json` | ground-truth CSV |
| 9–11 | Open LLM from Hugging Face (Qwen2.5-14B-Instruct on an A100): incident report + validator, and question answering with tools | No API key; about 30 GB download |
| 12 | Download everything as a zip | |

**Before you start:** go to *Runtime → Change runtime type → A100 GPU*. Then run the cells top to bottom.
Every number this notebook prints is **measured** on your clips. Copy those numbers into the report.

## 1 · Setup

In [ ]:
!nvidia-smi -L || echo "No GPU found - Runtime > Change runtime type > A100 GPU (T4 or CPU work, with smaller models)"
!pip -q install ultralytics shapely lap
import torch
print("CUDA available:", torch.cuda.is_available())

## 2 · Load videos
With `USE_BENCHMARK = True` nothing is uploaded here: step 3b downloads annotated footage instead. To use your own clips, set it to `False`.
Upload works for small clips (< ~100 MB). For bigger files, put them in Google Drive and set `USE_DRIVE = True`.

In [ ]:
import glob, os

USE_BENCHMARK = True   # True: step 3b downloads real, hand-annotated footage (MOT15 + MOT16); False: use your own clips
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/sentryscribe/videos"   # folder with your clips
VIDEO_DIR = "/content/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

if USE_BENCHMARK:
    print("Benchmark mode: the clips are downloaded in step 3b.")
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO_DIR = DRIVE_DIR
else:
    from google.colab import files
    for name, data in files.upload().items():
        with open(os.path.join(VIDEO_DIR, name), "wb") as f:
            f.write(data)

VIDEOS = sorted(p for ext in ("mp4", "avi", "mov", "mkv", "MP4", "MOV")
                for p in glob.glob(os.path.join(VIDEO_DIR, f"*.{ext}")))
print(len(VIDEOS), "videos:", [os.path.basename(v) for v in VIDEOS])

## 3 · Scene and rule configuration
Zone and line coordinates are **fractions of the frame** (0–1, origin top-left), so one scene works at any resolution.
Run step 5 to see them drawn on your video, then adjust and re-run. To give one clip its own scene, add it to `SCENES`.

Zone `type`: `restricted` raises intrusion events, `pedestrian_only` raises vehicle-in-zone events, and `watch` raises only loitering and crowding.
For a line, `wrong_way` is the crossing direction to flag (`up`/`down` for flat lines, `left`/`right` for steep ones).

In [ ]:
def pick_detector():
    # Bigger detector and input size on bigger GPUs: crowded scenes with small, far people need both.
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    if gb >= 35:
        return "yolov8x.pt", 1280      # A100: COCO mAP 53.9
    if gb >= 14:
        return "yolov8m.pt", 960       # T4 / L4: mAP 50.2
    return "yolov8n.pt", 640           # CPU: mAP 37.3


DETECTOR, IMGSZ = pick_detector()      # or set them yourself, e.g. "yolov8s.pt", 640

CONFIG = dict(
    model=DETECTOR,              # YOLOv8 weights, downloaded on first use
    imgsz=IMGSZ,                 # input size; larger finds smaller people and bags, but is slower
    target_fps=10,               # frames analysed per second of video
    conf=0.20,                   # detector threshold (low so bags survive)
    person_vehicle_conf=0.35,    # stricter threshold for people and vehicles
    intrusion_s=1.0,             # ZONE_INTRUSION min duration
    veh_ped_s=2.0,               # VEHICLE_IN_PED_ZONE min duration
    loiter_s=60.0,               # LOITERING dwell (lower it, e.g. 20, for short staged clips)
    crowd_n=10, crowd_s=5.0,     # CROWDING: >= N people for >= S seconds (lower N for staged clips)
    abandon_s=30.0,              # ABANDONED_OBJECT: bag still and alone this long
    abandon_radius=0.08,         # "alone" = no person within this fraction of frame width
    stationary_tol=0.03,         # "still" = moved less than this fraction of frame width ...
    stationary_window_s=10.0,    # ... over this window
    cooldown_s=2.0,              # condition must be false this long before an event closes
    line_debounce_s=2.0,         # ignore repeat crossings of one line by one track within this
    min_track_obs=3,             # tracks seen in fewer frames are ignored in counts and line crossings
    stitch_gap_s=1.5,            # a new track ID appearing within this time ...
    stitch_dist=0.05,            # ... and this fraction of frame width of a lost same-class track continues it (0 = off)
    stitch_min_obs=10,           # only tracks seen this many frames can be continued (stops phantom detections chaining)
    smooth_alpha=0.5,            # position smoothing per track (1 = off)
    line_margin=0.01,            # a track must be this fraction of frame width clear of a line to count as on one side
    attend_min_obs=10,           # only people tracked this many frames count as attending a bag
)

DEFAULT_SCENE = {
    "zones": [
        {"name": "Restricted", "type": "restricted",
         "polygon": [[0.60, 0.45], [0.95, 0.45], [0.95, 0.95], [0.60, 0.95]]},
        {"name": "Walkway", "type": "pedestrian_only",
         "polygon": [[0.05, 0.55], [0.45, 0.55], [0.45, 0.98], [0.05, 0.98]]},
    ],
    "lines": [
        {"name": "Entry line", "p1": [0.05, 0.50], "p2": [0.95, 0.50], "wrong_way": "up"},
    ],
}
SCENES = {}   # e.g. {"gate_cam_01.mp4": {...same shape as DEFAULT_SCENE...}}

OUT_DIR = "/content/outputs"
print("Detector:", CONFIG["model"], "at", CONFIG["imgsz"], "px")

## 3b · Real-footage benchmark (MOT15 + MOT16)
Runs only when `USE_BENCHMARK = True` in step 2. It downloads [MOT15](https://motchallenge.net/data/MOT15/) (1.3 GB) and
[MOT16](https://motchallenge.net/data/MOT16/) (1.9 GB), public multi-object tracking benchmarks in which every person in every frame
of the training videos is boxed by hand. Zones are fixed in the image, so all nine **static-camera** training sequences are used
(the moving-camera ones are left out):

| Sequence | Scene | Size, frame rate | Length |
|---|---|---|---|
| `PETS09-S2L1` | Elevated surveillance camera over a campus walkway | 768×576, 7 fps | 114 s |
| `Venice-2` | Busy public square | 1920×1080, 30 fps | 20 s |
| `ADL-Rundle-6` | Pedestrian street | 1920×1080, 30 fps | 17.5 s |
| `KITTI-17` | Street corner from a parked car | 1224×370, 10 fps | 14.5 s |
| `TUD-Stadtmitte` | Pedestrian zone, eye level | 640×480, 25 fps | 7 s |
| `TUD-Campus` | Campus path, eye level | 640×480, 25 fps | 3 s |
| `MOT16-02` | Square with many people | 1920×1080, 30 fps | 20 s |
| `MOT16-04` | Elevated camera over a busy pedestrian street at night | 1920×1080, 30 fps | 35 s |
| `MOT16-09` | Shopping street entrance | 1920×1080, 30 fps | 17.5 s |

For each sequence the cell:
1. converts the image frames to an `.mp4` in `/content/videos/`;
2. places zones from where the annotated people actually walk: a *Restricted* zone (the part of the walked area that the fewest people enter, but at least two), a *Plaza* watch zone over the whole walked area, and an *Entry line* across its middle. Step 5 draws them;
3. writes the **ground truth** to `/content/ground_truth.csv`: the event definitions applied to the hand-annotated tracks, with no detector or tracker involved. Step 8 then measures how much real detection and tracking errors cost.

Only people are annotated, so this checks intrusion, loitering, line crossing and crowding; abandoned bags and vehicles are covered by the simulation study.
A vehicle the system flags here counts as a false alarm. Because the clips are short, loitering is 10 s and crowding uses the busiest 10% of each scene's frames.

In [ ]:
# [benchmark]
import configparser, csv, json, shutil, subprocess, zipfile
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
from shapely.geometry import LineString, Point, Polygon

# Every static-camera training sequence of MOT15 and MOT16 (zones are fixed in the image, so moving cameras are left out).
BENCH_SETS = {
    "MOT15": ("https://motchallenge.net/data/MOT15.zip", "1.3 GB",
              ["PETS09-S2L1", "Venice-2", "ADL-Rundle-6", "KITTI-17", "TUD-Stadtmitte", "TUD-Campus"]),
    "MOT16": ("https://motchallenge.net/data/MOT16.zip", "1.9 GB", ["MOT16-02", "MOT16-04", "MOT16-09"]),
}
BENCH_SEQS = [s for _, _, seqs in BENCH_SETS.values() for s in seqs]
BENCH_DIR = "/content/mot"
BENCH_CONFIG = dict(loiter_s=10.0, crowd_s=3.0)                    # the clips are only 3-114 s long
SEQ_FPS = {"PETS09-S2L1": 7, "KITTI-17": 10, "TUD-Stadtmitte": 25, "TUD-Campus": 25}   # used when a sequence has no seqinfo.ini


def fetch_mot(name, url, size, seqs):
    if all(glob.glob(f"{BENCH_DIR}/**/train/{s}/gt/gt.txt", recursive=True) for s in seqs):
        return
    zpath = f"/content/{name}.zip"
    if not os.path.exists(zpath):
        print(f"Downloading {name} ({size}), a few minutes ...")
        subprocess.run(["wget", "-q", "-O", zpath, url], check=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(f"{BENCH_DIR}/{name}", [n for n in z.namelist() if any(f"train/{s}/" in n for s in seqs)])


def seq_info(seq):
    d = glob.glob(f"{BENCH_DIR}/**/train/{seq}", recursive=True)[0]
    frames = sorted(glob.glob(f"{d}/img1/*.jpg"))
    h, w = cv2.imread(frames[0]).shape[:2]
    fps = SEQ_FPS.get(seq, 30)
    if os.path.exists(f"{d}/seqinfo.ini"):
        ini = configparser.ConfigParser()
        ini.read(f"{d}/seqinfo.ini")
        fps = ini["Sequence"].get("frameRate", fps)
    return d, len(frames), w, h, float(fps)


def load_gt_tracks(d, fps, has_class):
    # gt.txt rows start: frame, id, left, top, width, height, conf.
    # MOT15 (10 columns): conf 0 marks boxes to ignore; columns 8-10 are world coordinates, -1 when unknown.
    # MOT16 (9 columns): column 8 is the class. Keep every person the detector would see (walking 1, on a bicycle 2,
    # standing or sitting 7), whatever conf says, and drop the rest (vehicles, reflections, ...).
    tracks = defaultdict(list)
    with open(f"{d}/gt/gt.txt") as f:
        for row in csv.reader(f):
            if len(row) < 7:
                continue
            frame, tid, x, y, bw, bh, conf = (float(v) for v in row[:7])
            if not (float(row[7]) in (1, 2, 7) if has_class else conf != 0):
                continue
            tracks[int(tid)].append(((frame - 1) / fps, (x + bw / 2, y + bh)))   # (video time, foot point)
    for v in tracks.values():
        v.sort()
    return tracks


def auto_scene(tracks, w, h, fps, cfg):
    # Zones from where the annotated people actually walk, so every sequence has events to find.
    feet = np.array([p for v in tracks.values() for _, p in v]) / [w, h]
    x0, x1 = np.percentile(feet[:, 0], [5, 95])
    y0, y1 = np.percentile(feet[:, 1], [5, 95])
    cw, ch = (x1 - x0) / 3, (y1 - y0) / 3
    need = cfg["intrusion_s"] * fps                       # frames inside that make an intrusion
    cells = []
    for i in range(3):
        for j in range(3):
            a, b = x0 + i * cw, y0 + j * ch
            n = sum(sum(a * w <= p[0] <= (a + cw) * w and b * h <= p[1] <= (b + ch) * h for _, p in v) >= need
                    for v in tracks.values())
            cells.append((n, a, b))
    entered = [c for c in cells if c[0] >= 2]
    _, a, b = min(entered) if entered else max(cells)    # least-visited cell that at least two people enter

    def rect(a, b, c, d):
        a, b, c, d = (round(float(v), 3) for v in (a, b, c, d))
        return [[a, b], [c, b], [c, d], [a, d]]

    plaza = Polygon([(x * w, y * h) for x, y in rect(x0, y0, x1, y1)])
    people = Counter(t for v in tracks.values() for t, p in v if plaza.contains(Point(p)))
    crowd_n = max(3, int(np.percentile(list(people.values()), 90)))   # the busiest 10% of frames count as a crowd
    ym = round(float(np.median(feet[:, 1])), 3)
    return {
        "zones": [dict(name="Restricted", type="restricted", polygon=rect(a, b, a + cw, b + ch)),
                  dict(name="Plaza", type="watch", polygon=rect(x0, y0, x1, y1))],
        "lines": [dict(name="Entry line", p1=[round(float(x0), 3), ym], p2=[round(float(x1), 3), ym], wrong_way="up")],
        "config": dict(BENCH_CONFIG, crowd_n=crowd_n),
    }


def runs(times, gap):
    # Group sorted times into [start, end] runs; a gap longer than `gap` starts a new run.
    out = []
    for t in times:
        if out and t - out[-1][1] <= gap:
            out[-1][1] = t
        else:
            out.append([t, t])
    return out


def label_events(tracks, scene, w, h, fps, cfg, video):
    # Ground truth: the event definitions applied to the hand-annotated tracks, with no detector or tracker.
    zones = [(z["name"], z["type"], Polygon([(x * w, y * h) for x, y in z["polygon"]])) for z in scene["zones"]]
    gap = max(cfg["cooldown_s"], 1 / fps) + 1e-6          # an event ends once its condition is false for cooldown_s
    rows = []

    def add(etype, zone, start, end):
        rows.append(dict(video=video, type=etype, zone=zone, start_s=round(start, 2), end_s=round(end, 2)))

    crowd = defaultdict(Counter)
    for v in tracks.values():
        inside = defaultdict(list)
        for t, p in v:
            for name, _, poly in zones:
                if poly.contains(Point(p)):
                    inside[name].append(t)
                    crowd[name][t] += 1
        for name, ztype, _ in zones:
            for s, e in runs(inside[name], gap):
                if ztype == "restricted" and e - s >= cfg["intrusion_s"]:
                    add("ZONE_INTRUSION", name, s, e)
                if e - s >= cfg["loiter_s"]:
                    add("LOITERING", name, s, e)
    for name, counts in crowd.items():
        for s, e in runs(sorted(t for t, n in counts.items() if n >= cfg["crowd_n"]), gap):
            if e - s >= cfg["crowd_s"]:
                add("CROWDING", name, s, e)
    for ln in scene["lines"]:
        geom = LineString([(ln["p1"][0] * w, ln["p1"][1] * h), (ln["p2"][0] * w, ln["p2"][1] * h)])
        for v in tracks.values():
            last = -1e9
            for (t0, a), (t1, b) in zip(v, v[1:]):
                if t1 - t0 <= 1.0 and a != b and LineString([a, b]).intersects(geom) and t1 - last > cfg["line_debounce_s"]:
                    add("LINE_CROSSING", ln["name"], t1, t1)
                    last = t1
    return rows


if USE_BENCHMARK:
    for name, (url, size, seqs) in BENCH_SETS.items():
        fetch_mot(name, url, size, seqs)
    SCENES.clear()
    VIDEOS, gt_rows = [], []
    for seq in BENCH_SEQS:
        d, n_frames, w, h, fps = seq_info(seq)
        mp4 = f"{VIDEO_DIR}/{seq}.mp4"
        if not os.path.exists(mp4):
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", f"{fps:g}", "-i", f"{d}/img1/%06d.jpg",
                            "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-c:v", "libx264", "-pix_fmt", "yuv420p", mp4], check=True)
        tracks = load_gt_tracks(d, fps, has_class=seq in BENCH_SETS["MOT16"][2])
        assert tracks, f"{seq}: no annotated people read from {d}/gt/gt.txt"
        scene = SCENES[f"{seq}.mp4"] = auto_scene(tracks, w, h, fps, CONFIG)
        rows = label_events(tracks, scene, w, h, fps, {**CONFIG, **scene["config"]}, f"{seq}.mp4")
        VIDEOS.append(mp4)
        gt_rows += rows
        print(f"{seq}: {n_frames} frames, {w}x{h} at {fps:g} fps, {len(tracks)} annotated people, "
              f"crowd = {scene['config']['crowd_n']}+ people | ground truth: {dict(Counter(r['type'] for r in rows))}")

    GT_CSV = "/content/ground_truth.csv"
    pd.DataFrame(gt_rows, columns=["video", "type", "zone", "start_s", "end_s"]).to_csv(GT_CSV, index=False)
    os.makedirs(OUT_DIR, exist_ok=True)
    shutil.copy(GT_CSV, f"{OUT_DIR}/ground_truth.csv")
    with open(f"{OUT_DIR}/benchmark_scenes.json", "w") as f:
        json.dump(SCENES, f, indent=2)
    print(f"\n{len(gt_rows)} ground-truth events written to {GT_CSV}")
else:
    print("Benchmark off: using your own clips and the scenes from step 3.")

## 4 · Pipeline code
Run this cell; it only defines functions. It holds the frame loop (detect → track → rules), the event engine with hysteresis, and the JSON and video writers.

In [ ]:
# [engine]
import json, math, subprocess, time
from collections import Counter, defaultdict, deque
from pathlib import Path

import cv2
import numpy as np
from shapely.geometry import LineString, Point, Polygon

PERSON = {"person"}
VEHICLES = {"car", "truck", "bus", "motorcycle", "bicycle"}
MOTOR_VEHICLES = {"car", "truck", "bus", "motorcycle"}
BAGS = {"backpack", "handbag", "suitcase"}
COCO_IDS = [0, 1, 2, 3, 5, 7, 24, 26, 28]   # the classes above, as COCO ids


def scene_for(video_path):
    return SCENES.get(Path(video_path).name, DEFAULT_SCENE)


def scene_to_px(scene, w, h):
    # Convert 0-1 scene coordinates into pixel geometry.
    zones = [dict(z, poly=Polygon([(x * w, y * h) for x, y in z["polygon"]]))
             for z in scene.get("zones", [])]
    lines = []
    for ln in scene.get("lines", []):
        p1 = (ln["p1"][0] * w, ln["p1"][1] * h)
        p2 = (ln["p2"][0] * w, ln["p2"][1] * h)
        lines.append(dict(ln, p1=p1, p2=p2, geom=LineString([p1, p2]),
                          horizontal=abs(p2[0] - p1[0]) >= abs(p2[1] - p1[1])))
    return zones, lines


ZONE_COLORS = {"restricted": (0, 0, 255), "pedestrian_only": (0, 200, 255), "watch": (255, 200, 0)}


def draw_scene(frame, zones, lines):
    for z in zones:
        pts = np.array(z["poly"].exterior.coords, np.int32)
        color = ZONE_COLORS.get(z["type"], (255, 200, 0))
        cv2.polylines(frame, [pts], True, color, 2)
        cv2.putText(frame, z["name"], (int(pts[0][0]) + 4, int(pts[0][1]) + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    for ln in lines:
        p1 = tuple(int(v) for v in ln["p1"])
        p2 = tuple(int(v) for v in ln["p2"])
        cv2.line(frame, p1, p2, (255, 0, 255), 2)
        cv2.putText(frame, ln["name"], (p1[0] + 4, p1[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    return frame


class EventEngine:
    # Opens an event once a condition has held for min_dur, closes it after cooldown_s without it.

    def __init__(self, video_id, snap_dir, cooldown, first_id=1):
        self.video_id, self.cooldown, self.next_id = video_id, cooldown, first_id
        self.snap_dir = Path(snap_dir)
        self.snap_dir.mkdir(parents=True, exist_ok=True)
        self.pending, self.events = {}, []

    def _open(self, etype, start, info, frame):
        eid = f"E{self.next_id}"
        self.next_id += 1
        snap = None
        if frame is not None:
            snap = str(self.snap_dir / f"{eid}.jpg")
            cv2.imwrite(snap, frame)
        ev = dict(event_id=eid, video_id=self.video_id, type=etype, object_class=info["object_class"],
                  track_ids=list(info["track_ids"]), zone=info["zone"], start_ts=round(start, 2),
                  end_ts=None, confidence=round(float(info["conf"]), 3), snapshot=snap,
                  details=dict(info["details"]))
        self.events.append(ev)
        return ev

    def instant(self, etype, t, info, frame):
        ev = self._open(etype, t, info, frame)
        ev["end_ts"] = ev["start_ts"]

    def update(self, t, active, frame):
        for key, info in active.items():
            p = self.pending.get(key)
            if p is None:               # "since" lets a rule backdate the start to when its condition really began
                p = self.pending[key] = dict(start=info.get("since", t), last=t, confs=[], ids=set(), max_count=0, event=None)
            p["last"] = t
            p["confs"].append(info["conf"])
            p["ids"].update(info["track_ids"])
            p["max_count"] = max(p["max_count"], info["details"].get("count", 0))
            if p["event"] is None and t - p["start"] >= info["min_dur"]:
                p["event"] = self._open(key[0], p["start"], info, frame)
        for key in [k for k in self.pending if k not in active]:
            if t - self.pending[key]["last"] > self.cooldown:
                self._close(self.pending.pop(key))

    def _close(self, p):
        ev = p["event"]
        if ev is None:          # condition never lasted min_dur: no event
            return
        ev["end_ts"] = round(p["last"], 2)
        ev["confidence"] = round(float(np.mean(p["confs"])), 3)
        ev["track_ids"] = sorted(p["ids"])
        if p["max_count"]:
            ev["details"]["max_count"] = p["max_count"]
            ev["details"].pop("count", None)
        ev["details"]["duration_s"] = round(ev["end_ts"] - ev["start_ts"], 2)

    def finish(self):
        for p in self.pending.values():
            self._close(p)
        self.pending.clear()
        self.events.sort(key=lambda e: e["start_ts"])


class TrackBook:
    # Per-track history: (t, foot point) positions plus summary metadata.

    def __init__(self, maxlen):
        self.hist = defaultdict(lambda: deque(maxlen=maxlen))
        self.meta = {}
        self.last_cross = {}
        self.alias = {}         # tracker ID -> stitched ID
        self.alone_since = {}   # bag track -> time it was last left alone
        self.side = {}          # (track, line) -> (committed side, position when committed)


def line_offset(ln, p):
    # Signed distance (px) from point p to a counting line, or None if p is beyond the line's ends.
    (x1, y1), (x2, y2) = ln["p1"], ln["p2"]
    dx, dy = x2 - x1, y2 - y1
    length2 = dx * dx + dy * dy
    u = ((p[0] - x1) * dx + (p[1] - y1) * dy) / length2
    if not 0 <= u <= 1:
        return None
    return ((p[0] - x1) * dy - (p[1] - y1) * dx) / math.sqrt(length2)


def stitch_ids(t, dets, book, cfg, frame_w):
    # Map a brand-new tracker ID onto a same-class track that vanished moments ago at the same place,
    # so an ID switch doesn't reset dwell timers or create duplicate events.
    out = [(book.alias.get(d[0], d[0]),) + tuple(d[1:]) for d in dets]
    if cfg.get("stitch_gap_s", 0) <= 0:
        return out
    taken = {d[0] for d in out}
    for k, (tid, name, conf, box) in enumerate(out):
        if tid in book.meta:
            continue
        foot = ((box[0] + box[2]) / 2, box[3])
        best = None
        for old, m in book.meta.items():
            if (old in taken or m["obs"] < cfg["stitch_min_obs"] or not 0 < t - m["last"] <= cfg["stitch_gap_s"]
                    or m["classes"].most_common(1)[0][0] != name):
                continue
            d = math.dist(book.hist[old][-1][1], foot)
            if d <= cfg["stitch_dist"] * frame_w and (best is None or d < best[0]):
                best = (d, old)
        if best:
            book.alias[tid] = best[1]
            taken.add(best[1])
            out[k] = (best[1], name, conf, box)
    return out


def analyse_frame(t, dets, zones, lines, book, engine, frame, cfg, frame_w):
    # dets: list of (track_id, class_name, conf, (x1, y1, x2, y2)) for one frame at time t (seconds).
    dets = stitch_ids(t, dets, book, cfg, frame_w)
    active = {}
    persons_in_zone = defaultdict(list)
    person_feet = []
    for tid, name, conf, (x1, y1, x2, y2) in dets:
        foot = ((x1 + x2) / 2, y2)                      # bottom-centre = ground contact point
        prev = book.hist[tid][-1][1] if book.hist[tid] else None
        a = cfg["smooth_alpha"]
        if prev is not None and a < 1:                  # exponential smoothing removes box jitter
            foot = (prev[0] + a * (foot[0] - prev[0]), prev[1] + a * (foot[1] - prev[1]))
        book.hist[tid].append((t, foot))
        m = book.meta.setdefault(tid, dict(classes=Counter(), first=t, last=t, confs=[], zones=set(), obs=0))
        m["classes"][name] += 1
        m["last"], m["obs"] = t, m["obs"] + 1
        m["confs"].append(conf)
        inside = [z for z in zones if z["poly"].contains(Point(foot))]
        m["zones"].update(z["name"] for z in inside)
        base = dict(object_class=name, track_ids=[tid], conf=conf, details={})

        for z in inside:
            if z["type"] == "restricted" and name in PERSON | VEHICLES:
                active[("ZONE_INTRUSION", z["name"], tid)] = dict(base, zone=z["name"], min_dur=cfg["intrusion_s"])
            if z["type"] == "pedestrian_only" and name in MOTOR_VEHICLES:
                active[("VEHICLE_IN_PED_ZONE", z["name"], tid)] = dict(base, zone=z["name"], min_dur=cfg["veh_ped_s"])
            if name in PERSON:
                active[("LOITERING", z["name"], tid)] = dict(base, zone=z["name"], min_dur=cfg["loiter_s"])
                persons_in_zone[z["name"]].append((tid, conf))
        if name in PERSON and m["obs"] >= cfg["attend_min_obs"]:
            person_feet.append(foot)

        if name in PERSON | VEHICLES:
            margin = cfg["line_margin"] * frame_w
            for ln in lines:
                d = line_offset(ln, foot)
                if d is None or abs(d) < margin:        # off the line's extent, or inside the dead band
                    continue
                key = (tid, ln["name"])
                side = 1 if d > 0 else -1
                committed = book.side.get(key)
                book.side[key] = (side, foot)
                if committed is None or committed[0] == side or m["obs"] < cfg["min_track_obs"]:
                    continue
                if t - book.last_cross.get(key, -1e9) <= cfg["line_debounce_s"]:
                    continue
                book.last_cross[key] = t
                before = committed[1]
                if ln["horizontal"]:
                    direction = "down" if foot[1] > before[1] else "up"
                else:
                    direction = "right" if foot[0] > before[0] else "left"
                info = dict(base, zone=ln["name"],
                            details={"direction": direction, "wrong_way": direction == ln.get("wrong_way")})
                engine.instant("LINE_CROSSING", t, info, frame)

    tol = cfg["stationary_tol"] * frame_w
    radius = cfg["abandon_radius"] * frame_w
    for tid, name, conf, _ in dets:
        if name not in BAGS:
            continue
        foot = book.hist[tid][-1][1]
        still_since = t
        for tt, p in reversed(book.hist[tid]):
            if math.dist(p, foot) > tol:
                break
            still_since = tt
        if any(math.dist(p, foot) <= radius for p in person_feet):
            book.alone_since.pop(tid, None)
            continue
        alone_since = book.alone_since.setdefault(tid, t)
        if t - still_since >= cfg["stationary_window_s"]:
            active[("ABANDONED_OBJECT", None, tid)] = dict(object_class=name, track_ids=[tid], conf=conf, zone=None,
                                                          details={}, min_dur=cfg["abandon_s"],
                                                          since=max(still_since, alone_since))

    for zname, members in persons_in_zone.items():
        if len(members) >= cfg["crowd_n"]:
            active[("CROWDING", zname, None)] = dict(
                object_class="person", track_ids=[m[0] for m in members], zone=zname,
                conf=float(np.mean([m[1] for m in members])), details={"count": len(members)},
                min_dur=cfg["crowd_s"])

    engine.update(t, active, frame)


def track_summaries(book, video_id):
    return [dict(video_id=video_id, track_id=tid, object_class=m["classes"].most_common(1)[0][0],
                 first_ts=round(m["first"], 2), last_ts=round(m["last"], 2), obs=m["obs"],
                 zones_visited=sorted(m["zones"]), mean_conf=round(float(np.mean(m["confs"])), 3))
            for tid, m in book.meta.items()]


def to_h264(src, dst):
    # Re-encode for browser playback; keeps the mp4v file if ffmpeg is unavailable.
    try:
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-vcodec", "libx264",
                            "-pix_fmt", "yuv420p", str(dst)], capture_output=True)
    except FileNotFoundError:
        return src
    if r.returncode == 0:
        Path(src).unlink()
        return dst
    return src


def process_video(video_path, cfg, out_root, first_id=1, save_video=True):
    from ultralytics import YOLO
    import torch

    video_path = Path(video_path)
    vid = video_path.stem
    out = Path(out_root) / vid
    out.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    stride = max(1, round(fps / cfg["target_fps"]))
    zones, lines = scene_to_px(scene_for(video_path), w, h)
    engine = EventEngine(vid, out / "snapshots", cfg["cooldown_s"], first_id=first_id)
    book = TrackBook(maxlen=int(cfg["target_fps"] * max(cfg["stationary_window_s"], 5)) + 5)
    raw_path = out / "annotated_raw.mp4"
    writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps / stride, (w, h)) if save_video else None

    model = YOLO(cfg["model"])   # fresh model per video, so tracker IDs don't carry over between clips
    results = model.track(source=str(video_path), stream=True, persist=True, tracker="bytetrack.yaml",
                          classes=COCO_IDS, conf=cfg["conf"], imgsz=cfg["imgsz"], vid_stride=stride, verbose=False)
    t0, n = time.time(), 0
    for i, r in enumerate(results):
        t = ((i + 1) * stride - 1) / fps        # Ultralytics returns the last frame of each stride group
        dets = []
        b = r.boxes
        if b is not None and b.id is not None:
            for tid, c, cf, box in zip(b.id.int().tolist(), b.cls.int().tolist(), b.conf.tolist(), b.xyxy.tolist()):
                name = r.names[c]
                if name not in BAGS and cf < cfg["person_vehicle_conf"]:
                    continue
                dets.append((tid, name, cf, tuple(box)))
        frame = draw_scene(r.plot(line_width=2), zones, lines)
        cv2.putText(frame, f"t={t:7.1f}s", (10, h - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        analyse_frame(t, dets, zones, lines, book, engine, frame, cfg, w)
        if writer is not None:
            writer.write(frame)
        n = i + 1
        if n % 50 == 0:
            print(f"  {vid}: {t:7.1f}s processed", end="\r")
    elapsed = time.time() - t0
    engine.finish()

    video_file = None
    if writer is not None:
        writer.release()
        video_file = str(to_h264(raw_path, out / "annotated.mp4"))

    tracks = track_summaries(book, vid)
    real_tracks = [tr for tr in tracks if tr["obs"] >= cfg["min_track_obs"]]
    duration = n_frames / fps if n_frames > 0 else n * stride / fps
    summary = dict(
        video_id=vid, file=video_path.name, duration_s=round(duration, 1), fps=round(fps, 2),
        resolution=[w, h], stride=stride, frames_processed=n, wall_time_s=round(elapsed, 1),
        processing_fps=round(n / elapsed, 2) if elapsed > 0 else None,
        realtime_factor=round(duration / elapsed, 2) if elapsed > 0 else None,
        model=cfg["model"], device="cuda" if torch.cuda.is_available() else "cpu",
        unique_tracks_by_class=dict(Counter(tr["object_class"] for tr in real_tracks)),
        events_by_type=dict(Counter(e["type"] for e in engine.events)),
        annotated_video=video_file, config=cfg,
    )
    for fname, obj in (("events.json", engine.events), ("tracks.json", tracks), ("summary.json", summary)):
        with open(out / fname, "w") as f:
            json.dump(obj, f, indent=2)
    return summary, engine.events, tracks

## 5 · Preview zones on the first frame
The grid lines are at every 0.1 of the frame, so you can read off coordinates for `CONFIG`/`SCENES`.

In [ ]:
import matplotlib.pyplot as plt

for v in VIDEOS:
    cap = cv2.VideoCapture(v)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        print("Could not read", v)
        continue
    h, w = frame.shape[:2]
    for k in range(1, 10):
        cv2.line(frame, (int(w * k / 10), 0), (int(w * k / 10), h), (200, 200, 200), 1)
        cv2.line(frame, (0, int(h * k / 10)), (w, int(h * k / 10)), (200, 200, 200), 1)
        cv2.putText(frame, f"{k/10:.1f}", (int(w * k / 10) + 2, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        cv2.putText(frame, f"{k/10:.1f}", (2, int(h * k / 10) - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    zones, lines = scene_to_px(scene_for(v), w, h)
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(draw_scene(frame, zones, lines), cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(v))
    plt.axis("off")
    plt.show()

## 6 · Process all videos
Writes `events.json`, `tracks.json`, `summary.json`, `annotated.mp4` and `snapshots/` for each clip into `/content/outputs/<clip>/`.

In [ ]:
ALL_EVENTS, ALL_TRACKS, SUMMARIES = [], [], []
for v in VIDEOS:
    print("Processing", os.path.basename(v))
    cfg = {**CONFIG, **scene_for(v).get("config", {})}     # a scene may override thresholds (benchmark clips do)
    s, ev, tr = process_video(v, cfg, OUT_DIR, first_id=len(ALL_EVENTS) + 1)
    SUMMARIES.append(s); ALL_EVENTS.extend(ev); ALL_TRACKS.extend(tr)
    print(f"  done: {s['frames_processed']} frames in {s['wall_time_s']} s "
          f"({s['processing_fps']} fps, {s['realtime_factor']}x real time) | events: {s['events_by_type']}")

with open(f"{OUT_DIR}/all_events.json", "w") as f:
    json.dump(ALL_EVENTS, f, indent=2)
with open(f"{OUT_DIR}/all_summaries.json", "w") as f:
    json.dump(SUMMARIES, f, indent=2)
print(f"\n{len(ALL_EVENTS)} events from {len(SUMMARIES)} videos")

## 7 · Inspect results

In [ ]:
import pandas as pd
from IPython.display import Image, Video, display

def hms(s):
    s = int(round(s))
    return f"{s // 3600:02d}:{s % 3600 // 60:02d}:{s % 60:02d}"

if ALL_EVENTS:
    df = pd.DataFrame(ALL_EVENTS)
    df["start"] = df["start_ts"].map(hms)
    df["end"] = df["end_ts"].map(lambda x: hms(x) if x is not None else "")
    display(df[["event_id", "video_id", "type", "object_class", "zone", "start", "end", "track_ids", "confidence"]])
    for e in ALL_EVENTS[:6]:
        if e["snapshot"]:
            print(e["event_id"], e["type"], e["zone"], hms(e["start_ts"]))
            display(Image(e["snapshot"], width=640))
else:
    print("No events. Check the zone preview, or lower the thresholds in CONFIG for short clips.")

In [ ]:
# Play one annotated video inline (embedding big files makes the notebook slow, so this stops at 60 MB)
first = SUMMARIES[0]["annotated_video"] if SUMMARIES else None
if first and os.path.getsize(first) < 60e6:
    display(Video(first, embed=True, width=800))
else:
    print("Download it from", first)

## 8 · Evaluate against ground truth
In benchmark mode, step 3b has already written `/content/ground_truth.csv`, so just run the cells below.

For your own clips, watch each one and write down when each event *really* started. Save the result as `ground_truth.csv`:

```
video,type,zone,start_s,end_s
clip1.mp4,ZONE_INTRUSION,Restricted,12.0,20.5
clip1.mp4,LINE_CROSSING,Entry line,31.0,31.0
clip2.mp4,ABANDONED_OBJECT,,40.0,95.0
```
`type` takes the event type names used above. `zone` is the zone or line name, left blank for `ABANDONED_OBJECT`.
A prediction counts as correct when its video, type and zone match a ground-truth row and its start is within `TOL_S` seconds of that row's start. Only videos that appear in the CSV are scored.

In [ ]:
GT_CSV = "/content/ground_truth.csv"
if not os.path.exists(GT_CSV):
    from google.colab import files
    up = files.upload()            # choose your ground_truth.csv
    GT_CSV = "/content/" + next(iter(up))

In [ ]:
# [eval]
import datetime


def norm_zone(z):
    if z is None or (isinstance(z, float) and math.isnan(z)):
        return ""
    return str(z).strip()


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else None
    r = tp / (tp + fn) if tp + fn else None
    f = 2 * p * r / (p + r) if p and r else (0.0 if p is not None and r is not None else None)
    return p, r, f


def evaluate(pred_events, gt_rows, tol_s=3.0, videos=None):
    # videos: the clips that were fully annotated. Default: the clips named in the ground truth.
    gt = [dict(g, video_id=Path(str(g["video"])).stem, zone=norm_zone(g.get("zone"))) for g in gt_rows]
    scored = set(videos) if videos is not None else {g["video_id"] for g in gt}
    gt = [g for g in gt if g["video_id"] in scored]
    preds = [e for e in pred_events if e["video_id"] in scored]
    per_type, all_err, totals = {}, [], Counter()
    for etype in sorted({g["type"] for g in gt} | {e["type"] for e in preds}):
        P = [e for e in preds if e["type"] == etype]
        G = [g for g in gt if g["type"] == etype]
        pairs = sorted((abs(p["start_ts"] - float(g["start_s"])), pi, gi)
                       for pi, p in enumerate(P) for gi, g in enumerate(G)
                       if p["video_id"] == g["video_id"] and norm_zone(p["zone"]) == g["zone"]
                       and abs(p["start_ts"] - float(g["start_s"])) <= tol_s)
        used_p, used_g, err = set(), set(), []
        for d, pi, gi in pairs:             # greedy: closest pairs first, one-to-one
            if pi not in used_p and gi not in used_g:
                used_p.add(pi); used_g.add(gi); err.append(d)
        tp, fp, fn = len(err), len(P) - len(err), len(G) - len(err)
        p, r, f = prf(tp, fp, fn)
        per_type[etype] = dict(tp=tp, fp=fp, fn=fn, precision=p, recall=r, f1=f,
                               mean_abs_start_err_s=round(float(np.mean(err)), 2) if err else None)
        totals.update(tp=tp, fp=fp, fn=fn)
        all_err += err
    p, r, f = prf(totals["tp"], totals["fp"], totals["fn"])
    overall = dict(tp=totals["tp"], fp=totals["fp"], fn=totals["fn"], precision=p, recall=r, f1=f,
                   mean_abs_start_err_s=round(float(np.mean(all_err)), 2) if all_err else None)
    return dict(overall=overall, per_type=per_type, videos_scored=sorted(scored), tolerance_s=tol_s)


def throughput(summaries):
    return dict(
        device=sorted({s["device"] for s in summaries}), model=sorted({s["model"] for s in summaries}),
        mean_processing_fps=round(float(np.mean([s["processing_fps"] for s in summaries])), 2),
        mean_realtime_factor=round(float(np.mean([s["realtime_factor"] for s in summaries])), 2),
        total_video_s=round(sum(s["duration_s"] for s in summaries), 1),
        total_wall_s=round(sum(s["wall_time_s"] for s in summaries), 1),
    )

In [ ]:
TOL_S = 3.0
gt_rows = pd.read_csv(GT_CSV).to_dict("records")
SCORED = [Path(v).stem for v in VIDEOS] if USE_BENCHMARK else None   # benchmark clips are all fully annotated
METRICS = evaluate(ALL_EVENTS, gt_rows, tol_s=TOL_S, videos=SCORED)
METRICS["per_video"] = {v: evaluate(ALL_EVENTS, gt_rows, tol_s=TOL_S, videos=[v])["overall"] for v in METRICS["videos_scored"]}
METRICS["throughput"] = throughput(SUMMARIES)
METRICS["measured_at"] = datetime.datetime.now().isoformat(timespec="seconds")
METRICS["config"] = CONFIG
with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(METRICS, f, indent=2)

table = pd.DataFrame(METRICS["per_type"]).T
table.loc["OVERALL"] = pd.Series(METRICS["overall"])
display(table.round(3))
display(pd.DataFrame(METRICS["per_video"]).T.round(3))
print("Throughput:", METRICS["throughput"])

## 9 · LLM setup
Loads an open instruction-tuned model from Hugging Face onto the Colab GPU: no API key, no account, and nothing leaves the machine.
It picks the largest Qwen2.5 model that fits the GPU:

| GPU | Model | Download |
|---|---|---|
| A100 80 GB | `Qwen/Qwen2.5-32B-Instruct` | ~65 GB |
| **A100 40 GB** | **`Qwen/Qwen2.5-14B-Instruct`** | ~30 GB |
| L4 24 GB | `Qwen/Qwen2.5-7B-Instruct` | ~15 GB |
| T4 16 GB | `Qwen/Qwen2.5-3B-Instruct` | ~6 GB |

All of them support tool calling. To force one, set `LLM_MODEL` by hand. The LLM only receives the event JSON, never video frames.

In [ ]:
!pip -q install transformers accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def pick_llm():
    # Largest Qwen2.5 instruct model whose half-precision weights fit this GPU, leaving room for YOLO and the KV cache.
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    for need_gb, name in [(75, "Qwen/Qwen2.5-32B-Instruct"),   # A100 80 GB: ~65 GB of weights
                          (35, "Qwen/Qwen2.5-14B-Instruct"),   # A100 40 GB: ~30 GB
                          (20, "Qwen/Qwen2.5-7B-Instruct")]:   # L4 24 GB: ~15 GB
        if gb >= need_gb:
            return name
    return "Qwen/Qwen2.5-3B-Instruct"                           # T4 16 GB or CPU: ~6 GB


LLM_MODEL = pick_llm()        # or set a name yourself, e.g. "Qwen/Qwen2.5-7B-Instruct"
dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if torch.cuda.is_available() else torch.float32
print("Loading", LLM_MODEL, "in", dtype, "(first run downloads the weights from Hugging Face)")
tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto", torch_dtype=dtype)
llm.eval()
print("Loaded", LLM_MODEL, "on", llm.device)

In [ ]:
# [llm]
import re


def generate(chat, tools=None, max_tokens=1200):
    # One reply from the local model for a chat (list of role/content dicts). Greedy decoding, so runs are repeatable.
    prompt = tok.apply_chat_template(chat, tools=tools, add_generation_prompt=True, tokenize=False)
    ids = tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=max_tokens, do_sample=False)
    text = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)
    return text.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()


def off_script(text):
    # Share of letters outside the Latin script. Qwen sometimes drifts into Thai or Chinese mid-answer.
    letters = [ch for ch in text if ch.isalpha()]
    return sum(ord(ch) > 0x24F for ch in letters) / len(letters) if letters else 0.0


def call_llm(system, messages, tools=None, max_tokens=1200):
    # Returns (text, tool_calls). The model asks for a tool by writing <tool_call>{"name": ..., "arguments": {...}}</tool_call>.
    chat = [{"role": "system", "content": system}] + messages
    raw = generate(chat, tools, max_tokens)
    if off_script(raw) > 0.05:             # drifted out of English: ask once more
        raw = generate(chat + [{"role": "user", "content": "Reply in English only."}], tools, max_tokens)
    calls = []
    for blob in re.findall(r"<tool_call>(.*?)</tool_call>", raw, re.S):
        try:
            c = json.loads(blob)
        except json.JSONDecodeError:
            continue
        args = c.get("arguments") or {}
        calls.append(dict(name=c.get("name"), arguments=json.loads(args) if isinstance(args, str) else args))
    return re.sub(r"<tool_call>.*?</tool_call>", "", raw, flags=re.S).strip(), calls


def hms(s):
    s = int(round(s))
    return f"{s // 3600:02d}:{s % 3600 // 60:02d}:{s % 60:02d}"


def compact_event(e):
    out = dict(id=e["event_id"], video=e["video_id"], type=e["type"], object=e["object_class"], zone=e["zone"],
               start=hms(e["start_ts"]), end=hms(e["end_ts"]) if e["end_ts"] is not None else None,
               start_s=e["start_ts"], tracks=e["track_ids"], conf=e["confidence"])
    out.update({k: v for k, v in e["details"].items() if k in ("direction", "wrong_way", "max_count", "duration_s")})
    return out


REPORT_SYSTEM = '''You write security incident reports from an automated CCTV event log.
Rules:
- Use ONLY facts in the event log JSON. Do not invent people, descriptions, causes or events.
- Cite every factual sentence with the event ids it relies on, in square brackets, e.g. [E3] or [E3][E7].
- Write times as HH:MM:SS from the start of the video, exactly as given in the log.
- Objects are anonymous track numbers; never guess identity, gender, age or intent.
- If a category of event did not occur, say "not observed". If the log has no events, say nothing was observed and cite nothing.
- Write in English.
Output Markdown with these sections: ## Summary, ## Timeline, ## Notable incidents, ## Statistics, ## Recommended follow-up.'''


def validate_report(text, events):
    known = {e["event_id"] for e in events}
    cited = [eid for group in re.findall(r"\[([^\]]+)\]", text) for eid in re.findall(r"E\d+", group)]
    unknown = sorted(set(cited) - known, key=lambda x: int(x[1:]))
    windows = [(e["start_ts"] - 2, (e["end_ts"] if e["end_ts"] is not None else e["start_ts"]) + 2) for e in events]
    bad_times = []
    for ts in set(re.findall(r"\b(\d{2}):(\d{2}):(\d{2})\b", text)):
        sec = int(ts[0]) * 3600 + int(ts[1]) * 60 + int(ts[2])
        if not any(a <= sec <= b for a, b in windows):
            bad_times.append(":".join(ts))
    return dict(citations=len(cited), unique_cited=len(set(cited)), unknown_ids=unknown,
                citation_validity=round(1 - len([c for c in cited if c not in known]) / len(cited), 3) if cited else None,
                times_outside_events=sorted(bad_times), valid=not unknown and (bool(cited) or not events))


def generate_report(summary, events):
    digest = json.dumps(dict(video=summary["file"], duration=hms(summary["duration_s"]),
                             unique_tracks_by_class=summary["unique_tracks_by_class"],
                             events=[compact_event(e) for e in events]), separators=(",", ":"))
    messages = [{"role": "user", "content": f"Event log:\n{digest}\n\nWrite the incident report."}]
    text, _ = call_llm(REPORT_SYSTEM, messages)
    check = validate_report(text, events)
    if not check["valid"]:                 # one retry with the validator's complaint
        fix = ("Rewrite the report citing only event ids that exist in the log." if events else
               "The log has no events. Rewrite the report saying nothing was observed, with no event ids.")
        messages += [{"role": "assistant", "content": text},
                     {"role": "user", "content": f"Validation failed: {json.dumps(check)}. {fix}"}]
        text, _ = call_llm(REPORT_SYSTEM, messages)
        check = validate_report(text, events)
        check["retried"] = True
    return text, check


# ---- query agent: the LLM can only reach the data through these functions ----

def _in_window(start, end, t_from, t_to):
    end = start if end is None else end
    return (t_from is None or end >= t_from) and (t_to is None or start <= t_to)


# A small model's arguments are loose ("restricted zone", "PETS09-S2L1.mp4", "00:01:10", "vehicle"),
# so the tools normalise them instead of silently matching nothing.

def _secs(v):
    if v is None or isinstance(v, (int, float)):
        return v
    parts = [float(x) for x in str(v).strip().split(":")]
    return sum(x * 60 ** i for i, x in enumerate(reversed(parts)))


def _name(s):
    return re.sub(r"\b(zone|line|area)\b", "", str(s or "").lower()).strip(" _-")


def _zone_ok(want, have):
    w, h = _name(want), _name(have)
    return not w or (bool(h) and (w in h or h in w))


def _video_ok(want, have):
    return want is None or Path(str(want)).stem.lower() == str(have).lower()


def _type_of(t):
    t = str(t or "").upper().replace(" ", "_")
    return next((e for e in EVENT_TYPES if t == e or t in e), t) if t else None


def _class_ok(want, have):
    w = str(want or "").lower()
    groups = {"vehicle": VEHICLES, "vehicles": VEHICLES, "bag": BAGS, "bags": BAGS, "people": PERSON, "persons": PERSON}
    return not w or have in groups.get(w, {w, w.rstrip("s")})


def search_events(type=None, object_class=None, zone=None, video_id=None, t_from=None, t_to=None, limit=20):
    etype, t_from, t_to = _type_of(type), _secs(t_from), _secs(t_to)
    rows = [e for e in ALL_EVENTS
            if (etype is None or e["type"] == etype) and _class_ok(object_class, e["object_class"])
            and _zone_ok(zone, e["zone"]) and _video_ok(video_id, e["video_id"])
            and _in_window(e["start_ts"], e["end_ts"], t_from, t_to)]
    return dict(total=len(rows), events=[compact_event(e) for e in rows[:int(limit or 20)]])


def count_objects(object_class, zone=None, video_id=None, t_from=None, t_to=None):
    t_from, t_to = _secs(t_from), _secs(t_to)
    rows = [tr for tr in ALL_TRACKS
            if _class_ok(object_class, tr["object_class"]) and tr["obs"] >= CONFIG["min_track_obs"]
            and (zone is None or any(_zone_ok(zone, z) for z in tr["zones_visited"]))
            and _video_ok(video_id, tr["video_id"]) and _in_window(tr["first_ts"], tr["last_ts"], t_from, t_to)]
    return dict(object_class=object_class, unique_tracks=len(rows),
                note="Unique tracker IDs; one real object can occasionally get two IDs after occlusion.")


def get_event(event_id):
    for e in ALL_EVENTS:
        if e["event_id"].lower() == str(event_id).strip("[] ").lower():
            return dict(compact_event(e), snapshot=e["snapshot"], details=e["details"])
    return dict(error=f"no event {event_id}")


def video_summary(video_id=None):
    keep = ("video_id", "file", "duration_s", "unique_tracks_by_class", "events_by_type")
    return [{k: s[k] for k in keep} for s in SUMMARIES if _video_ok(video_id, s["video_id"])]


EVENT_TYPES = ["ZONE_INTRUSION", "LOITERING", "CROWDING", "LINE_CROSSING", "ABANDONED_OBJECT", "VEHICLE_IN_PED_ZONE"]
CLASSES = sorted(PERSON | VEHICLES | BAGS)
_window = {"t_from": {"type": "number", "description": "Start of time window, seconds from video start"},
           "t_to": {"type": "number", "description": "End of time window, seconds from video start"}}


def _tool(name, description, properties, required=()):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": properties, "required": list(required)}}}


TOOLS = [
    _tool("search_events", "Find detected events, filtered by type, object class, zone or line name, video and time window. Returns at most `limit` events plus the total count.",
          {"type": {"type": "string", "enum": EVENT_TYPES}, "object_class": {"type": "string", "enum": CLASSES},
           "zone": {"type": "string", "description": "Zone or line name"}, "video_id": {"type": "string"},
           **_window, "limit": {"type": "integer"}}),
    _tool("count_objects", "Count unique tracked objects of one class, optionally only those that entered a zone, within a video or time window.",
          {"object_class": {"type": "string", "enum": CLASSES}, "zone": {"type": "string"},
           "video_id": {"type": "string"}, **_window}, ["object_class"]),
    _tool("get_event", "Full details for one event id such as E12.", {"event_id": {"type": "string"}}, ["event_id"]),
    _tool("video_summary", "Duration, object counts and event counts per video. Omit video_id for all videos.",
          {"video_id": {"type": "string"}}),
]
TOOL_FNS = dict(search_events=search_events, count_objects=count_objects, get_event=get_event, video_summary=video_summary)

QUERY_SYSTEM = '''You answer questions about CCTV footage. Your only source of truth is the tools, which query an automated event log.
Always call a tool before answering. Present times as HH:MM:SS from the start of the video and cite event ids like [E3].
Event types: zone intrusion = ZONE_INTRUSION, loitering = LOITERING, line crossing = LINE_CROSSING, unattended bag = ABANDONED_OBJECT,
crowd = CROWDING, vehicle in a pedestrian zone = VEHICLE_IN_PED_ZONE. Use only the filters the question needs: a question that
names no video is about all videos, so leave out video_id; leave out zone and the time window unless the question gives them. If a search returns total 0, say it was not observed.
Never guess identities or describe appearance. Answer in English.'''


def query_context():
    # The real video ids and zone/line names, so the model's filters match the log.
    names = sorted({x["name"] for s in SUMMARIES for k in ("zones", "lines") for x in scene_for(s["file"]).get(k, [])})
    return f"\nVideos: {', '.join(s['video_id'] for s in SUMMARIES)}. Zones and lines: {', '.join(names)}."


def ask(question, max_rounds=6, trace=None):
    # trace, if given, collects every tool call with its result count, so a wrong answer can be traced to its query.
    messages = [{"role": "user", "content": question}]
    system = QUERY_SYSTEM + query_context()
    for _ in range(max_rounds):
        text, calls = call_llm(system, messages, tools=TOOLS, max_tokens=800)
        if not calls:
            return text
        messages.append({"role": "assistant", "content": text,
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            try:
                out = TOOL_FNS[c["name"]](**c["arguments"])
            except Exception as ex:          # unknown tool or bad arguments: tell the model instead of crashing
                out = {"error": f"{type(ex).__name__}: {ex}"}
            messages.append({"role": "tool", "name": c["name"], "content": json.dumps(out)})
            if trace is not None:
                trace.append(dict(tool=c["name"], arguments=c["arguments"],
                                  total=out.get("total", out.get("unique_tracks")) if isinstance(out, dict) else len(out)))
    return "Stopped: too many tool rounds."

## 10 · Incident reports
Generates one report per video, then checks every cited event id and time against the log. The checks are saved next to each report.

In [ ]:
from IPython.display import Markdown

REPORT_CHECKS = {}
for s in SUMMARIES:
    events = [e for e in ALL_EVENTS if e["video_id"] == s["video_id"]]
    t0 = time.time()
    text, check = generate_report(s, events)
    check["latency_s"] = round(time.time() - t0, 1)
    check["model"] = LLM_MODEL
    REPORT_CHECKS[s["video_id"]] = check
    out = Path(OUT_DIR) / s["video_id"]
    (out / "report.md").write_text(text)
    (out / "report_check.json").write_text(json.dumps(check, indent=2))
    print(s["file"], "->", check)
    display(Markdown(text))

## 11 · Ask questions
Each question is about one event type, so its answer is **graded automatically against the event log**. An answer counts as correct when
it is in English, its search covered every video (the questions name none), and either it cites at least one event of that type and only
real ids, or, when the log has none, it cites nothing and says so.
This checks that answers are grounded in the log, not that counts or wording are right, so read the answers too.
Every tool call behind an answer is saved in `qa_answers.json`, so a wrong answer can be traced to the query that caused it.

In [ ]:
QUESTIONS = [   # (question, event type it is about; None = open question, not graded)
    ("Summarise what happened in each video.", None),
    ("Did anyone enter the restricted zone? When?", "ZONE_INTRUSION"),
    ("How many people crossed the entry line, and did anyone cross it the wrong way?", "LINE_CROSSING"),
    ("Did anyone loiter? Where and for how long?", "LOITERING"),
    ("Was there any crowding? When?", "CROWDING"),
    ("Was any bag left unattended? Where and for how long?", "ABANDONED_OBJECT"),
    ("Did any vehicle enter the walkway?", "VEHICLE_IN_PED_ZONE"),
]


def grade(answer, etype, trace):
    ids = {e["event_id"] for e in ALL_EVENTS if e["type"] == etype}
    cited = set(re.findall(r"\bE\d+\b", answer))
    all_videos = any(t["tool"] == "search_events" and not t["arguments"].get("video_id") for t in trace)
    if off_script(answer) > 0.05 or not all_videos:
        return False
    if ids:
        return bool(cited & ids) and cited <= {e["event_id"] for e in ALL_EVENTS}
    return not cited and bool(re.search(r"\b(no|not|none|never)\b", answer, re.I))


QA = []
for q, etype in QUESTIONS:
    t0, trace = time.time(), []
    a = ask(q, trace=trace)
    QA.append(dict(question=q, about=etype, answer=a, tool_calls=trace, latency_s=round(time.time() - t0, 1),
                   events_in_log=sum(e["type"] == etype for e in ALL_EVENTS) if etype else None,
                   correct=grade(a, etype, trace) if etype else None))
    print(f"Q: {q}\nA: {a}\ntools: {trace}\n({QA[-1]['latency_s']} s, correct: {QA[-1]['correct']})\n" + "-" * 80)
with open(f"{OUT_DIR}/qa_answers.json", "w") as f:
    json.dump(QA, f, indent=2)

In [ ]:
# Query accuracy over the graded questions. Disagree with a grade? Set QA[i]["correct"] by hand and re-run this cell.
marked = [x for x in QA if x["correct"] is not None]
if marked:
    METRICS = json.load(open(f"{OUT_DIR}/metrics.json")) if os.path.exists(f"{OUT_DIR}/metrics.json") else {}
    METRICS["query_accuracy"] = dict(correct=sum(x["correct"] for x in marked), total=len(marked),
                                     accuracy=round(sum(x["correct"] for x in marked) / len(marked), 3),
                                     mean_latency_s=round(float(np.mean([x["latency_s"] for x in QA])), 1))
    METRICS["report_checks"] = REPORT_CHECKS
    METRICS["llm_model"] = LLM_MODEL
    json.dump(METRICS, open(f"{OUT_DIR}/metrics.json", "w"), indent=2)
    json.dump(QA, open(f"{OUT_DIR}/qa_answers.json", "w"), indent=2)
    print(METRICS["query_accuracy"])
else:
    print("No graded answers.")

## 12 · Download outputs
The zip contains all outputs: events, tracks, summaries, snapshots, annotated videos, reports and `metrics.json`.

In [ ]:
!cd /content && zip -qr sentryscribe_outputs.zip outputs
from google.colab import files
files.download("/content/sentryscribe_outputs.zip")